In [ ]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

Larch port of `us/modeling_mnl.ipynb` (the Biogeme model), alongside `modeling_torch_choice.ipynb`
(the torch-choice port). Same choice structure: `alt=0` is staying, `alt=1..50` are the move
alternatives, and the data-prep cells below (reading, dtype cleanup, `ALT_CHOICE`, the melt to long
format, and the same-named-shared-column trick for tying coefficients across stay/move contexts) are
identical to the torch-choice port -- only the final "build the model and estimate it" section differs,
since that's the part where the two packages' APIs diverge. See `modeling_torch_choice.ipynb`'s intro
cell for the full list of fixes applied relative to `modeling_mnl.ipynb` as currently written on disk
(stale `OWN_RACE_PCT`/`OWN_GROUP_PCT` column names, a mangled NAICS variable name, the omitted
white/other-race terms, `ALT{i}_STATE` dtype).

**Larch's formula system (`P("name") * X("expression")`) makes the two awkward parts of the
torch-choice port unnecessary:**
- **Shared coefficients across stay/move contexts** are just `P("name")` referenced in the same
  `utility_ca` formula that gets evaluated once per `(person, alt)` row -- no need for the "same
  column name" trick to fake it (though the data is still built that way below, since the stay-context
  and move-context expressions are still different data, just tied to the same coefficient either way).
- **The un-parameterized `log(population)` offset** (coefficient pinned at 1, not estimated) is
  `m.lock_value("log_pop_offset", 1)` -- a native "holdfast" mechanism, not a `forward()` subclass hack.

**Package status, from the "experimental, not feature-complete" warning larch prints on import:** this
is `larch` v6 (the JAX/numba dual-backend rewrite), not the older, more battle-tested v5 line. Verified
empirically while porting -- **`m.utility_ca` accumulation via `+=` in a loop silently drops all but the
last term** (`m.utility_ca = PX(a); m.utility_ca += PX(b)` ends up as an *empty* `LinearFunction`, and
the model then estimates zero free parameters without erroring). Building the full sum on a plain local
variable first, then assigning it to `m.utility_ca` once, works correctly -- that's what's done below.
This isn't documented anywhere obvious, so worth flagging if you extend this notebook.

**Optimizer:** `maximize_loglike(method="BHHH")` -- Berndt-Hall-Hall-Hausman, a quasi-Newton method
built around the log-likelihood's own outer-product-of-gradients Hessian approximation. It's Larch's
default for unconstrained problems (SLSQP is the default only when finite parameter bounds/constraints
are set, which this model doesn't use) and is standard for MLE of logit-family models -- much closer in
spirit to Biogeme's BFGS than torch-choice's installed-version-limited Adam.


In [ ]:
import larch as lx
import numpy as np
import pandas as pd
from larch import PX


In [ ]:
year = 2018
num_alternatives = 50


### Read data, only keep used columns

In [ ]:
# same columns= restriction technique as modeling_mnl.ipynb: read only the columns V[0]/V[i]
# actually reference, instead of the full ~3,800-column file, since most of it is unused census
# fields. ALT_VARYING_SUFFIXES covers the ALT{i}_<suffix> destination columns V[i] uses;
# INDIV_COLS covers everything else, plus CHOSEN/STAY/ALT{i}_PUMA needed to build ALT_CHOICE below.
# NOTE: OWN_RACE_ETH_PCT / OWN_NAICS_GROUP_PCT here, not OWN_RACE_PCT / OWN_GROUP_PCT as modeling_mnl.ipynb's
# cell 5 currently has -- see modeling_torch_choice.ipynb's intro cell for why.
ALT_VARYING_SUFFIXES = [
    "ALT_COMMUTE_PCT",
    "CBSA",
    "COLLEGE_PCT",
    "DIST",
    "ENT_JOBS_PCT",
    "FOREIGN_BORN_PCT",
    "HH_MED_INC",
    "HH_WITH_CHILD_PCT",
    "HOUSE_VACANCY_PCT",
    "MED_HOUSE_VALUE_OVER_MED_HH_INC",
    "MED_RENT_PCT_HH_INC",
    "MED_TRAVEL_TIME",
    "MIL_PCT",
    "OWN_AGE_PCT",
    "OWN_NAICS_GROUP_PCT",
    "OWN_RACE_ETH_PCT",
    "STATE",
    "TOT_POP",
    "TYPE",
    "UNEMP_RATE",
]

INDIV_COLS = [
    "CHOSEN",
    "STAY",
    "AAPI",
    "AGE_18_22",
    "AGE_18_34",
    "AGE_23_29",
    "AGE_30_39",
    "AGE_35_64",
    "AGE_40_49",
    "AGE_50_64",
    "AGE_OVER_65",
    "BLACK",
    "CHILD",
    "CHILD_6_TO_17",
    "CHILD_UNDER_6",
    "EDU_BACHELORS",
    "EDU_HIGH",
    "EDU_NOHIGH",
    "FOREIGN",
    "House vacancy proportion.ORIG",
    "INDIAN",
    "IN_COLLEGE",
    "IN_MILITARY",
    "LATINO",
    "MARRIED_MORE_THAN_YEAR",
    "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG",
    "Median gross rent as a percentage of household income.ORIG",
    "Median house value over median household income.ORIG",
    "Median travel time.ORIG",
    "NAICS_AGR_EXT",
    "NAICS_GOODS_TRADE",
    "NAICS_GOVT",
    "NAICS_GROUP_PCT_AGR_EXT.ORIG",
    "NAICS_GROUP_PCT_GOODS_TRADE.ORIG",
    "NAICS_GROUP_PCT_GOVT.ORIG",
    "NAICS_GROUP_PCT_HIGH_ED.ORIG",
    "NAICS_GROUP_PCT_LICENSE.ORIG",
    "NAICS_HIGH_ED",
    "NAICS_LICENSE",
    "NAME_NUM.ORIG",
    "POBP",
    "Proportion alternative commute.ORIG",
    "Proportion foreign born.ORIG",
    "Proportion of households with children.ORIG",
    "Proportion of people 18-34.ORIG",
    "Proportion of people 35-64.ORIG",
    "Proportion of people 65+.ORIG",
    "Proportion of people AAPI.ORIG",
    "Proportion of people Black.ORIG",
    "Proportion of people Indian.ORIG",
    "Proportion of people Latino.ORIG",
    "Proportion of people in college.ORIG",
    "Proportion of people in military.ORIG",
    "RECENTLY_MARRIED",
    "RECENTLY_WIDOWED_OR_DIVORCED",
    "SINGLE_PARENT",
    "ST",
    "TYPE_NUM.ORIG",
    "Unemployment rate.ORIG",
    "WORK1_MAR",
    "WORK2_MAR",
    "Total Population.Total Population.SE_A00001_001.ORIG",
    "Proportion of entertainment jobs.ORIG",
]

needed_cols = (
    INDIV_COLS
    + [f"ALT{i}_PUMA" for i in range(1, num_alternatives + 1)]
    + [
        f"ALT{i}_{suf}"
        for i in range(1, num_alternatives + 1)
        for suf in ALT_VARYING_SUFFIXES
    ]
)
needed_cols = list(dict.fromkeys(needed_cols))
print(f"reading {len(needed_cols)} columns")

df = pd.read_parquet(f"../data/us_estdata_{year}.parquet", columns=needed_cols)
print(df.shape)


### Clean data, make everything numeric

In [ ]:
cat_cols = df.dtypes[df.dtypes == "category"].keys()
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype(int))

obj_cols = list(df.dtypes[df.dtypes == "object"].keys())
df[obj_cols] = df[obj_cols].apply(lambda x: x.astype(int))

# ALT{i}_STATE is pandas "string" dtype in the current parquet (not category/object, so the two
# conversions above miss it) -- cast explicitly so it's comparable to the int64 ST/POBP columns
# in the same_state / birthstate terms below. CHOSEN and ALT{i}_PUMA are also "string" dtype, but
# stay that way on purpose: they're only ever compared to each other, both as strings.
state_cols = [f"ALT{i}_STATE" for i in range(1, num_alternatives + 1)]
df[state_cols] = df[state_cols].astype(int)

df["person_id"] = np.arange(len(df))


### Creating alternatives

In [ ]:
# defining the chosen alternative for each person explicitly, same convention as modeling_mnl.ipynb:
# each person has chosen from alternatives 0-num_alternatives, 0 represents staying and 1-num_alternatives
# represent moving to the PUMAs each represents. Movers match their true chosen destination by PUMA;
# stayers are always alt=0.
df["ALT_CHOICE"] = 0
for i in range(1, num_alternatives + 1):
    var = f"ALT{i}_PUMA"
    df["ALT_CHOICE"] = np.where(df[var] == df["CHOSEN"], i, df["ALT_CHOICE"])
df["ALT_CHOICE"] = np.where(df["STAY"] == 1, 0, df["ALT_CHOICE"])
assert (df["ALT_CHOICE"] > 0).sum() == (df["STAY"] == 0).sum(), (
    "every mover must match exactly one ALT{i}_PUMA"
)


### Reshape to long (idca) format

Larch's `Dataset.construct.from_idca` wants one row per `(caseid, altid)` pair, same shape as
torch-choice's long format -- so the reshape is identical to `modeling_torch_choice.ipynb`: a local
`wide_to_long` replacement for `xlogit.utils.wide_to_long` (xlogit is no longer a project dependency)
melts the `ALT{i}_<suffix>` destination columns, and the stay block (one row per person, no
alternative-varying columns) is built directly. Both get concatenated below.

Coefficient sharing between stay and move contexts (e.g. `proportion_same_age_18_34`, compared to the
*origin* on stay rows and to the *destination* on move rows) is expressed by giving both contexts'
values the same column name, same as the torch-choice port -- although here it's really just a
convenience for building the data; Larch would tie `P("proportion_same_age_18_34")` to the same
coefficient in `utility_ca` regardless of whether the underlying data came from one shared column or
two separately-named ones.

`c_proportion_same_race_white` and `c_proportion_same_race_other` from `modeling_mnl.ipynb` are omitted
here (see `modeling_torch_choice.ipynb`'s intro cell) -- their source columns aren't in `INDIV_COLS`,
and one is populated from a truncated column-name string in the source notebook.


In [ ]:
def wide_to_long(df, id_col, alt_list, alt_name, varying, sep, alt_is_prefix):
    assert alt_is_prefix and sep == "_"
    varying_cols = {f"{alt}{sep}{suf}" for alt in alt_list for suf in varying}
    id_vars = [c for c in df.columns if c not in varying_cols]
    frames = []
    for alt in alt_list:
        rename = {f"{alt}{sep}{suf}": suf for suf in varying}
        sub = df[id_vars + list(rename.keys())].rename(columns=rename)
        sub[alt_name] = alt
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)


move_long = wide_to_long(
    df,
    id_col="person_id",
    alt_list=[f"ALT{i}" for i in range(1, num_alternatives + 1)],
    alt_name="alt_label",
    varying=ALT_VARYING_SUFFIXES,
    sep="_",
    alt_is_prefix=True,
)
move_long["alt"] = move_long["alt_label"].str[len("ALT") :].astype(int)
move_long["choice"] = (move_long["alt"] == move_long["ALT_CHOICE"]).astype(int)

same_state = (move_long["ST"] == move_long["STATE"]).astype(float)
# NAME_NUM.ORIG and ALT{i}_CBSA are factorized against the same CBSA-name codebook
# (see create_estdata.ipynb), so they're directly comparable
same_cbsa = (move_long["NAME_NUM.ORIG"] == move_long["CBSA"]).astype(float)
same_type_t34 = (move_long["TYPE_NUM.ORIG"] == 0).astype(float)
same_type_metro = (move_long["TYPE_NUM.ORIG"] == 1).astype(float)
same_type_nonmetro = (move_long["TYPE_NUM.ORIG"] == 2).astype(float)
alt_type_t34 = (move_long["TYPE"] == 0).astype(float)
alt_type_metro = (move_long["TYPE"] == 1).astype(float)
alt_type_nonmetro = (move_long["TYPE"] == 2).astype(float)

# fixed (coefficient == 1, not estimated) log-population offset -- destination population for move rows.
move_long["log_pop_offset"] = np.log(move_long["TOT_POP"])

# move-only terms (structurally zero at alt=0, since staying isn't a move-type decision)
move_long["destchoice_logdist"] = np.log(move_long["DIST"] + 1)
move_long["destchoice_samecbsa"] = same_cbsa
move_long["destchoice_samestate"] = same_state
move_long["destchoice_birthstate"] = (move_long["POBP"] == move_long["STATE"]).astype(
    float
)
# origin area type x destination area type; nonmetro_nonmetro is the omitted reference category,
# matching the commented-out c_destchoice_nonmetro_nonmetro in modeling_mnl.ipynb's cell 22.
move_long["destchoice_T34_T34"] = same_type_t34 * alt_type_t34
move_long["destchoice_T34_metro"] = same_type_t34 * alt_type_metro
move_long["destchoice_T34_nonmetro"] = same_type_t34 * alt_type_nonmetro
move_long["destchoice_metro_T34"] = same_type_metro * alt_type_t34
move_long["destchoice_metro_metro"] = same_type_metro * alt_type_metro
move_long["destchoice_metro_nonmetro"] = same_type_metro * alt_type_nonmetro
move_long["destchoice_nonmetro_T34"] = same_type_nonmetro * alt_type_t34
move_long["destchoice_nonmetro_metro"] = same_type_nonmetro * alt_type_metro

# shared terms (move-context value: comparing the mover to the destination)
move_long["proportion_same_age_18_34"] = (
    move_long["AGE_18_34"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_same_age_35_64"] = (
    move_long["AGE_35_64"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_same_age_65_plus"] = (
    move_long["AGE_OVER_65"] * move_long["OWN_AGE_PCT"]
)
move_long["proportion_hh_with_children_if_have_children"] = (
    move_long["HH_WITH_CHILD_PCT"] * move_long["CHILD"]
)
move_long["proportion_college_if_in_college"] = (
    move_long["IN_COLLEGE"] * move_long["COLLEGE_PCT"]
)
move_long["proportion_foreign_if_foreign"] = (
    move_long["FOREIGN"] * move_long["FOREIGN_BORN_PCT"]
)
move_long["median_hh_income_in_tens_of_thousands"] = move_long["HH_MED_INC"] / 10_000
move_long["median_house_value_over_median_income"] = move_long[
    "MED_HOUSE_VALUE_OVER_MED_HH_INC"
]
move_long["median_gross_rent_percentage_hh_inc"] = move_long["MED_RENT_PCT_HH_INC"]
move_long["unemp_rate"] = move_long["UNEMP_RATE"]
move_long["vacancy_rate"] = move_long["HOUSE_VACANCY_PCT"]
move_long["median_travel_time"] = move_long["MED_TRAVEL_TIME"]
move_long["proportion_alt_commute"] = move_long["ALT_COMMUTE_PCT"]
move_long["proportion_ent"] = move_long["ENT_JOBS_PCT"]
move_long["proportion_ent_18_34"] = move_long["AGE_18_34"] * move_long["ENT_JOBS_PCT"]
move_long["proportion_ent_35_64"] = move_long["AGE_35_64"] * move_long["ENT_JOBS_PCT"]
move_long["proportion_also_mil"] = move_long["IN_MILITARY"] * move_long["MIL_PCT"]
# c_proportion_same_naics_goods_trade uses ALT{i}_OWN_NAICS_GROUP_PCT here, not the mangled
# f"{alt}OWN_GROUPOWN_NAICS_GROUP_PCT_PCT" that modeling_mnl.ipynb's cell 23 currently has --
# see modeling_torch_choice.ipynb's intro cell.
move_long["proportion_same_naics_govt"] = (
    move_long["NAICS_GOVT"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_goods_trade"] = (
    move_long["NAICS_GOODS_TRADE"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_license"] = (
    move_long["NAICS_LICENSE"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_high_ed"] = (
    move_long["NAICS_HIGH_ED"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_naics_agr_ext"] = (
    move_long["NAICS_AGR_EXT"] * move_long["OWN_NAICS_GROUP_PCT"]
)
move_long["proportion_same_race_black"] = (
    move_long["BLACK"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_same_race_aapi"] = (
    move_long["AAPI"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_same_race_indian"] = (
    move_long["INDIAN"] * move_long["OWN_RACE_ETH_PCT"]
)
move_long["proportion_also_latino"] = (
    move_long["LATINO"] * move_long["OWN_RACE_ETH_PCT"]
)

MOVE_ONLY_TERMS = [
    "destchoice_logdist",
    "destchoice_samecbsa",
    "destchoice_samestate",
    "destchoice_birthstate",
    "destchoice_T34_T34",
    "destchoice_T34_metro",
    "destchoice_T34_nonmetro",
    "destchoice_metro_T34",
    "destchoice_metro_metro",
    "destchoice_metro_nonmetro",
    "destchoice_nonmetro_T34",
    "destchoice_nonmetro_metro",
]
SHARED_TERMS = [
    "proportion_same_age_18_34",
    "proportion_same_age_35_64",
    "proportion_same_age_65_plus",
    "proportion_hh_with_children_if_have_children",
    "proportion_college_if_in_college",
    "proportion_foreign_if_foreign",
    "median_hh_income_in_tens_of_thousands",
    "median_house_value_over_median_income",
    "median_gross_rent_percentage_hh_inc",
    "unemp_rate",
    "vacancy_rate",
    "median_travel_time",
    "proportion_alt_commute",
    "proportion_ent",
    "proportion_ent_18_34",
    "proportion_ent_35_64",
    "proportion_also_mil",
    "proportion_same_naics_govt",
    "proportion_same_naics_goods_trade",
    "proportion_same_naics_license",
    "proportion_same_naics_high_ed",
    "proportion_same_naics_agr_ext",
    "proportion_same_race_black",
    "proportion_same_race_aapi",
    "proportion_same_race_indian",
    "proportion_also_latino",
]
move_long = move_long[
    ["person_id", "alt", "choice", "log_pop_offset"] + MOVE_ONLY_TERMS + SHARED_TERMS
]
print(move_long.shape)


### Stay block: one `alt=0` row per person, direct (no melt -- different variables entirely, not an
alternative-varying reshape)


In [ ]:
stay = df.copy()
stay["stay"] = 1.0  # c_stay: the stay alternative-specific constant
stay["stay_age_18_22"] = stay["AGE_18_22"]
stay["stay_age_23_29"] = stay["AGE_23_29"]
stay["stay_age_30_39"] = stay["AGE_30_39"]
stay["stay_age_40_49"] = stay["AGE_40_49"]
stay["stay_age_50_64"] = stay["AGE_50_64"]

stay["stay_child_under_6"] = stay["CHILD_UNDER_6"]
stay["stay_child_6_to_17"] = stay["CHILD_6_TO_17"]

stay["stay_married_more_than_year"] = stay["MARRIED_MORE_THAN_YEAR"]
stay["stay_married_less_than_year"] = stay["RECENTLY_MARRIED"]
stay["stay_recently_divorced_or_widowed"] = stay["RECENTLY_WIDOWED_OR_DIVORCED"]
stay["stay_2work_mar"] = stay["WORK2_MAR"]
stay["stay_single_parent"] = stay["SINGLE_PARENT"]

stay["stay_edu_college"] = stay["EDU_BACHELORS"]
stay["stay_edu_high"] = stay["EDU_HIGH"]

stay["stay_in_college"] = stay["IN_COLLEGE"]
stay["stay_foreign"] = stay["FOREIGN"]

# NOTE: this assumes that NAICS code stays constant between the origin and destination
stay["stay_mil"] = stay["IN_MILITARY"]
stay["stay_naics_govt"] = stay["NAICS_GOVT"]
stay["stay_naics_goods_trade"] = stay["NAICS_GOODS_TRADE"]
stay["stay_naics_license"] = stay["NAICS_LICENSE"]
stay["stay_naics_high_ed"] = stay["NAICS_HIGH_ED"]
stay["stay_naics_agr_ext"] = stay["NAICS_AGR_EXT"]

STAY_ONLY_TERMS = [
    "stay",
    "stay_age_18_22",
    "stay_age_23_29",
    "stay_age_30_39",
    "stay_age_40_49",
    "stay_age_50_64",
    "stay_child_under_6",
    "stay_child_6_to_17",
    "stay_married_more_than_year",
    "stay_married_less_than_year",
    "stay_recently_divorced_or_widowed",
    "stay_2work_mar",
    "stay_single_parent",
    "stay_edu_college",
    "stay_edu_high",
    "stay_in_college",
    "stay_foreign",
    "stay_mil",
    "stay_naics_govt",
    "stay_naics_goods_trade",
    "stay_naics_license",
    "stay_naics_high_ed",
    "stay_naics_agr_ext",
]

HH_INCOME_COL = (
    "Median Household Income (In 2018 Inflation Adjusted Dollars)."
    "Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
)

# fixed (coefficient == 1, not estimated) log-population offset -- origin population for the stay row.
stay["log_pop_offset"] = np.log(
    stay["Total Population.Total Population.SE_A00001_001.ORIG"]
)

# shared terms (stay-context value: comparing the person to their origin)
stay["proportion_same_age_18_34"] = (
    stay["Proportion of people 18-34.ORIG"] * stay["AGE_18_34"]
)
stay["proportion_same_age_35_64"] = (
    stay["Proportion of people 35-64.ORIG"] * stay["AGE_35_64"]
)
stay["proportion_same_age_65_plus"] = (
    stay["Proportion of people 65+.ORIG"] * stay["AGE_OVER_65"]
)
stay["proportion_hh_with_children_if_have_children"] = (
    stay["Proportion of households with children.ORIG"] * stay["CHILD"]
)
stay["proportion_college_if_in_college"] = (
    stay["Proportion of people in college.ORIG"] * stay["IN_COLLEGE"]
)
stay["proportion_foreign_if_foreign"] = (
    stay["Proportion foreign born.ORIG"] * stay["FOREIGN"]
)
stay["median_hh_income_in_tens_of_thousands"] = stay[HH_INCOME_COL] / 10_000
stay["median_house_value_over_median_income"] = stay[
    "Median house value over median household income.ORIG"
]
stay["median_gross_rent_percentage_hh_inc"] = stay[
    "Median gross rent as a percentage of household income.ORIG"
]
stay["unemp_rate"] = stay["Unemployment rate.ORIG"]
stay["vacancy_rate"] = stay["House vacancy proportion.ORIG"]
stay["median_travel_time"] = stay["Median travel time.ORIG"]
stay["proportion_alt_commute"] = stay["Proportion alternative commute.ORIG"]
stay["proportion_ent"] = stay["Proportion of entertainment jobs.ORIG"]
stay["proportion_ent_18_34"] = (
    stay["AGE_18_34"] * stay["Proportion of entertainment jobs.ORIG"]
)
stay["proportion_ent_35_64"] = (
    stay["AGE_35_64"] * stay["Proportion of entertainment jobs.ORIG"]
)
stay["proportion_also_mil"] = (
    stay["Proportion of people in military.ORIG"] * stay["IN_MILITARY"]
)
stay["proportion_same_naics_govt"] = (
    stay["NAICS_GROUP_PCT_GOVT.ORIG"] * stay["NAICS_GOVT"]
)
stay["proportion_same_naics_goods_trade"] = (
    stay["NAICS_GROUP_PCT_GOODS_TRADE.ORIG"] * stay["NAICS_GOODS_TRADE"]
)
stay["proportion_same_naics_license"] = (
    stay["NAICS_GROUP_PCT_LICENSE.ORIG"] * stay["NAICS_LICENSE"]
)
stay["proportion_same_naics_high_ed"] = (
    stay["NAICS_GROUP_PCT_HIGH_ED.ORIG"] * stay["NAICS_HIGH_ED"]
)
stay["proportion_same_naics_agr_ext"] = (
    stay["NAICS_GROUP_PCT_AGR_EXT.ORIG"] * stay["NAICS_AGR_EXT"]
)
stay["proportion_same_race_black"] = (
    stay["Proportion of people Black.ORIG"] * stay["BLACK"]
)
stay["proportion_same_race_aapi"] = (
    stay["Proportion of people AAPI.ORIG"] * stay["AAPI"]
)
stay["proportion_same_race_indian"] = (
    stay["Proportion of people Indian.ORIG"] * stay["INDIAN"]
)
stay["proportion_also_latino"] = (
    stay["Proportion of people Latino.ORIG"] * stay["LATINO"]
)

stay["alt"] = 0
stay["choice"] = (stay["ALT_CHOICE"] == 0).astype(int)
stay = stay[
    ["person_id", "alt", "choice", "log_pop_offset"] + STAY_ONLY_TERMS + SHARED_TERMS
]
print(stay.shape)


### Concatenate and build the Larch dataset

Each block only has real values for its own terms; the other block's terms are structurally `0` on its
rows (not missing). After sorting by `(person_id, alt)`, `Dataset.construct.from_idca` takes the long
dataframe directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the
torch-choice port.


In [ ]:
long_df = pd.concat([stay, move_long], ignore_index=True, sort=False)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS
long_df[varnames + ["log_pop_offset"]] = long_df[varnames + ["log_pop_offset"]].fillna(
    0.0
)
long_df = long_df.sort_values(["person_id", "alt"]).reset_index(drop=True)

num_persons = df["person_id"].nunique()
num_alts = num_alternatives + 1  # 51: alt=0 (stay) + alt=1..50 (move)
assert len(long_df) == num_persons * num_alts, (len(long_df), num_persons * num_alts)

idca = long_df.set_index(["person_id", "alt"])[["choice", "log_pop_offset"] + varnames]
ds = lx.Dataset.construct.from_idca(idca, crack=True)
ds


### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [ ]:
m = lx.Model(ds)
m.title = f"us_mnl_{year} (larch port of modeling_mnl.ipynb)"
m.compute_engine = "numba"

total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone (matches av[i] = 1 for all i in modeling_mnl.ipynb);
# no availability_ca_var needed.

m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


### Fitting

In [ ]:
print("null log-likelihood:", m.loglike())


In [ ]:
result = m.maximize_loglike(method="BHHH")
result


In [ ]:
m.calculate_parameter_covariance()
m.parameter_summary()
